# UEMR trên HT-Step: UniAV HT-Step + Omni HT-Step LoRA

Theo `docs/uemr/HTSTEP_UEMR_OMNI_UNIAV.md`. Mọi tham số, danh sách video, tiêu chí nằm trong `configs/htstep_uemr_protocol.json` (khoá, ô 3
kiểm sha256). **Không train, không chọn tham số bằng nhãn val, val chấm một lần.**

**Trước khi chạy:** chép thư mục `D:\Học\KL\output\htstep_uemr\uniav` (`uniav_raw.jsonl`, `checks.json`, `predictions_*.csv`) lên Drive vào
`MyDrive/Colab Notebooks/code KL/Fushion/uemr/htstep/uniav/a7c46353/`. UniAV đã chạy ở local (CPU, ô 6-7 chỉ kiểm hash và số tái lập).

Colab Secrets: `HF_TOKEN`. GPU: A100 (ô 9-10).

Vector student (`student_*.npz`) cũng nằm trong thư mục cache đó. Cột (S), (U), C1 dùng text của Omni thầy (`student` trong protocol): thầy của
UniAV khác `best`.

**Chạy thử trước:** `SMOKE = True` (ô 0, mặc định) chạy 5 video, ghi vào `smoke/` riêng, chỉ in CHECK, shape và tốc độ, không in chỉ số truy hồi; ô 13 tự
xoá `smoke/`. Xong đổi `SMOKE = False`, Run all để chạy cả kho.


In [ ]:
# ===== 0. Cấu hình =====
FUSHION_REF = 'cc50e2e8cb0b86940a98bb2d91df2f1b4459b6fd'   # commit trên nhánh htstep chứa configs/htstep_uemr_protocol.json
PROTO_SHA16 = '4c4ed5103c543b3f'   # sha256[:16] của protocol tại FUSHION_REF
SMOKE       = True       # True: chạy thử 5 video (chỉ CHECK, shape, tốc độ; ghi smoke/, ô 13 xoá). Xong đổi False rồi Run all
OMNI_TAGS   = ['best']   # thêm 'pt' nếu muốn mốc tham chiếu (encode thêm một lượt clip)
SEED        = 0
RUN_OMNI    = True       # ô 9-10
RUN_EVAL    = True       # ô 11-13
HANLI_REF, OMNI_FIX_REF, OMNI_AV_REF = '77fc4fa', 'ab772eb', '57a7f1f'   # code pin theo protocol (OMNI_AV: nhánh trích vector thầy)
FLAGS = {k: v for k, v in dict(globals()).items() if k.startswith('RUN_') or k in ('FUSHION_REF', 'PROTO_SHA16', 'OMNI_TAGS', 'SEED', 'SMOKE')}


In [ ]:
# ===== 1. Drive, code (Fushion, HanLi, Omni-fix), thư viện, unit test =====
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, sys, json, glob, shutil, subprocess, time, hashlib, platform, csv
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['HF_XET_CHUNK_CACHE_SIZE_BYTES'] = '0'
os.environ.pop('HF_XET_HIGH_PERFORMANCE', None)
T_START = time.time()
FUSHION, HANLI, OMNI_FIX, OMNI_AV = '/content/code/Fushion_UEMR', '/content/code/HanLi-Retriever', '/content/code/Omni-fix', '/content/code/Omni-av'
os.makedirs('/content/code', exist_ok=True)

def clone(url, path, ref, *branch):
    os.chdir('/content')      # Run all lần 2: cwd đang ở trong path sắp xoá thì git clone lỗi 128
    shutil.rmtree(path, ignore_errors=True)
    subprocess.run(['git', 'clone', '-q', *branch, url, path], check=True)
    r = subprocess.run(['git', '-C', path, 'checkout', '-q', ref], capture_output=True, text=True)
    assert r.returncode == 0, f'không checkout được {ref} của {url}: {r.stderr} (đã push chưa?)'
    return subprocess.run(['git', '-C', path, 'log', '-1', '--format=%h %s'], capture_output=True, text=True).stdout.strip()

COMMIT = clone('https://github.com/chien24/Fushion_UEMR.git', FUSHION, FUSHION_REF)
HANLI_COMMIT = clone('https://github.com/im-xiaoming/HanLi-Retriever.git', HANLI, HANLI_REF)
OMNI_COMMIT = clone('https://github.com/im-xiaoming/Omni-Retriever-Custom.git', OMNI_FIX, OMNI_FIX_REF, '-b', 'htstep_lora')
OMNI_AV_COMMIT = clone('https://github.com/im-xiaoming/Omni-Retriever-Custom.git', OMNI_AV, OMNI_AV_REF, '-b', 'htstep-av-extract')
print('Fushion :', COMMIT); print('HanLi   :', HANLI_COMMIT); print('Omni-fix:', OMNI_COMMIT); print('Omni-av :', OMNI_AV_COMMIT)
git = lambda *a: subprocess.run(['git', '-C', FUSHION, *a], capture_output=True, text=True).stdout.strip()
if FUSHION not in sys.path:
    sys.path.insert(0, FUSHION)
os.environ['PYTHONPATH'] = FUSHION + ':' + os.environ.get('PYTHONPATH', '')
os.chdir(FUSHION)

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout + r.stderr)[-3000:]
    if out.strip():
        print(out)
    if check and r.returncode:
        raise RuntimeError(f'lệnh lỗi ({r.returncode}): {cmd[:200]}')
    return r

def check(name, ok, detail=''):
    print(f"CHECK {name}: {'OK' if ok else 'FAIL'}" + (f' | {detail}' if detail else ''))
    assert ok, f'CHECK {name} FAIL: {detail}'

sh(f'pip install -q -r {FUSHION}/requirements.txt')
sh('pip uninstall -y -q torchao', check=False)
import numpy as np, pandas as pd, torch
from IPython.display import display
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'không có'
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0.0
ENV = {'python': platform.python_version(), 'torch': torch.__version__, 'numpy': np.__version__, 'gpu': gpu, 'gpu_GB': round(GPU_GB, 1),
       'fushion': COMMIT, 'hanli': HANLI_COMMIT, 'omni_fix': OMNI_COMMIT, 'omni_av': OMNI_AV_COMMIT, 'smoke': SMOKE}
print(json.dumps(ENV, indent=1, ensure_ascii=False))
r = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_uemr_htstep.py', '-q', '-p', 'no:cacheprovider'], cwd=FUSHION,
                   capture_output=True, text=True, env={**os.environ, 'UNIAV_REPO': HANLI})
print(r.stdout[-1200:])
check('unit test (gồm adapter UniAV thật)', r.returncode == 0 and ' skipped' not in r.stdout.splitlines()[-1], r.stdout.splitlines()[-1] if r.stdout else r.stderr[-300:])

def sha256(path, bs=1 << 24):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(bs), b''):
            h.update(chunk)
    return h.hexdigest()

def wj(path, obj):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + '.tmp'
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=1, ensure_ascii=False, default=str)
    os.replace(tmp, path)

def rj(path, default=None):
    return json.load(open(path, encoding='utf-8')) if os.path.exists(path) else default

DRIVE = '/content/drive/MyDrive'
ROOT  = f'{DRIVE}/Colab Notebooks/code KL/Fushion/uemr/htstep'
H     = f'{ROOT}/smoke' if SMOKE else ROOT      # mọi đầu ra của lượt này; cache UniAV luôn ở ROOT
LOCAL = '/content/smoke' if SMOKE else '/content'
if SMOKE:
    shutil.rmtree(H, ignore_errors=True); shutil.rmtree(LOCAL, ignore_errors=True)
VID, SEGDIR = f'{H}/videos_val', f'{H}/segments'
RUN_DRIVE = f'{DRIVE}/omniretriever/htstep_lora/lora_bs32'      # lượt train Omni
A = '/content/hts'
for d in (H, VID, SEGDIR, A, LOCAL):
    os.makedirs(d, exist_ok=True)


In [ ]:
# ===== 2. Tải model_best.pth.tar và adapter best; kiểm sha256 =====
from huggingface_hub import hf_hub_download
with open(f'{FUSHION}/configs/htstep_uemr_protocol.json', encoding='utf-8') as f:
    _P = json.load(f)
_M = _P['models']
UNIAV_CKPT = hf_hub_download(_M['uniav']['repo'], _M['uniav']['file'], local_dir=f'{A}/uniav_model')
check('sha256 model_best.pth.tar (UniAV)', sha256(UNIAV_CKPT) == _M['uniav']['sha256'], sha256(UNIAV_CKPT)[:16])
PT_ADAPTER, BEST_ADAPTER = '/content/adapters/omniretriever-7b', '/content/adapters/best'
if 'pt' in OMNI_TAGS or _P['student']['enabled'] or not os.path.exists(f'{RUN_DRIVE}/eval/adapter_best/adapter_model.safetensors'):
    sh(f"hf download {_M['omni_pt']['repo']} --include 'omniretriever-7b/*' --local-dir /content/adapters")
    check('sha256 adapter pt', sha256(f'{PT_ADAPTER}/adapter_model.safetensors') == _M['omni_pt']['sha256'])
if not os.path.exists(f'{BEST_ADAPTER}/adapter_model.safetensors'):
    os.makedirs(BEST_ADAPTER, exist_ok=True)
    src = f'{RUN_DRIVE}/eval/adapter_best'
    if os.path.exists(f'{src}/adapter_model.safetensors'):
        for f in os.listdir(src):
            shutil.copy2(f'{src}/{f}', f'{BEST_ADAPTER}/{f}')
    else:      # Drive chỉ còn LoRA-only: dựng lại y như eval_htstep.py
        sh(f'cd {OMNI_FIX} && PYTHONPATH={OMNI_FIX}/src python scripts/materialize_adapter.py {PT_ADAPTER} {RUN_DRIVE}/ckpt/best {BEST_ADAPTER}')
best_sha = sha256(f'{BEST_ADAPTER}/adapter_model.safetensors')
check('sha256 adapter best', best_sha == _M['omni_best']['sha256'], best_sha[:16])
TEACHER_ADAPTER = '/content/adapters/teacher'
if _P['student']['enabled']:      # đúng adapter thầy: LoRA của omni-adapters/checkpoint + head pt, sha256 cạnh best trong protocol
    T = _M['omni_teacher']
    teacher_lora = hf_hub_download(T['repo'], f"{T['dir']}/{T['file']}", local_dir=f'{A}/teacher')
    check('sha256 LoRA thầy', sha256(teacher_lora) == T['sha256'], sha256(teacher_lora)[:16])
    if not os.path.exists(f'{TEACHER_ADAPTER}/adapter_model.safetensors'):
        sh(f'cd {OMNI_FIX} && PYTHONPATH={OMNI_FIX}/src python scripts/materialize_adapter.py {PT_ADAPTER} {os.path.dirname(teacher_lora)} {TEACHER_ADAPTER}')
    teacher_sha = sha256(f'{TEACHER_ADAPTER}/adapter_model.safetensors')
    check('sha256 adapter thầy (LoRA + head pt) = protocol', teacher_sha == T['adapter_sha256'], teacher_sha[:16])
    print('best   :', best_sha); print('teacher:', teacher_sha)


In [ ]:
# ===== 3. Kiểm hash protocol =====
PROTO_PATH = f'{FUSHION}/configs/htstep_uemr_protocol.json'
PH = sha256(PROTO_PATH)
check('sha256 protocol', PH[:16] == PROTO_SHA16, f'tại {FUSHION_REF}: {PH[:16]}, notebook ghi {PROTO_SHA16}')
print('sha256 protocol:', PH)
PROTO_COMMIT = git('log', '-1', '--format=%h %ad', '--date=short', '--', 'configs/htstep_uemr_protocol.json')
check('protocol đã commit', bool(PROTO_COMMIT), PROTO_COMMIT)
PR = json.load(open(PROTO_PATH, encoding='utf-8'))
EV, CR, RESOLVED = PR['eval'], PR['criteria'], PR['resolved']
STUDENT = bool(PR['student']['enabled'])
BETA = float(PR['params']['beta'])
NB, BSEED = EV['bootstrap']['n_boot'], EV['bootstrap']['seed']
RES = f'{H}/results/{PROTO_SHA16[:8]}'
os.makedirs(RES, exist_ok=True)
print(json.dumps({'protocol': PH[:16], 'commit': PROTO_COMMIT, 'θ': RESOLVED['theta'], 'L': RESOLVED['chunk_sec'], 'student': STUDENT,
                  'kết quả': RES}, ensure_ascii=False))
MAN_PATH = f'{H}/run_manifest.json'
MAN = rj(MAN_PATH, {'runs': []})
RUN = {'started': time.strftime('%Y-%m-%d %H:%M:%S'), 'commit': COMMIT, 'protocol_sha16': PH[:16], 'flags': FLAGS, 'env': ENV, 'sec': {}, 'info': {}}
def done(part, t0, **info):
    RUN['sec'][part] = round(time.time() - t0, 1)
    RUN['info'][part] = info
    wj(MAN_PATH, {**MAN, 'runs': MAN['runs'] + [RUN]})
    print(f'[manifest] {part}: {RUN["sec"][part]} s')
from uemr_htstep import partitions as hp
LABEL = hp.ROWS
ROWS = list(LABEL)
INV = rj(f'{RES}/inventory.json', {})


In [ ]:
# ===== 4. Kho: lọc video, giao với train hai model = rỗng, câu hỏi, text trùng =====
from huggingface_hub import hf_hub_download
from uemr_htstep import data as hd
t0 = time.time()
D = PR['data']
HFC = D['val_manifest']['hf']
fetch_meta = lambda name: hf_hub_download(HFC, name, repo_type='dataset', local_dir=A)
val_p, train_p, items_p = fetch_meta(D['val_manifest']['file']), fetch_meta(D['train_manifest']['file']), fetch_meta(D['items_val']['file'])
for name, p, s in (('val manifest', val_p, D['val_manifest']['sha256']), ('train manifest', train_p, D['train_manifest']['sha256']),
                   ('items_val', items_p, D['items_val']['sha256'])):
    check(f'sha256 {name}', sha256(p) == s)
HTREPO = f'{A}/htstep_repo'
if not os.path.isdir(f'{HTREPO}/.git'):
    subprocess.run(['git', 'clone', '-q', D['annotations']['repo'], HTREPO], check=True)
subprocess.run(['git', '-C', HTREPO, 'checkout', '-q', D['annotations']['commit']], check=True)
ANN_P, TAX_P = f'{HTREPO}/{D["annotations"]["file"]}', f'{HTREPO}/{D["taxonomy"]["file"]}'
check('sha256 annotations.json chính thức', sha256(ANN_P) == D['annotations']['sha256'])
check('sha256 taxonomy.csv chính thức', sha256(TAX_P) == D['taxonomy']['sha256'])
check('sha256 configs/htstep_train_ids.json', sha256(f"{FUSHION}/{D['train_ids']['file']}") == D['train_ids']['sha256'])
ANN = json.load(open(ANN_P, encoding='utf-8'))
VAL_ROWS, TRAIN_ROWS = hd.read_jsonl(val_p), hd.read_jsonl(train_p)
IDS = json.load(open(f"{FUSHION}/{D['train_ids']['file']}", encoding='utf-8'))
omni_train = {r['video_id'] for r in TRAIN_ROWS}
check('danh sách train Omni = manifest', omni_train == set(IDS['omni_train']), f'{len(omni_train)} video')
check('danh sách train UniAV = protocol', len(IDS['uniav_train']) == D['train_ids']['uniav_train'], f"{len(IDS['uniav_train'])} video")
KHO, COUNTS = hd.select_gallery(ANN, D['gallery'], VAL_ROWS, IDS['uniav_train'], omni_train)
check('kho = protocol', KHO == D['gallery'] and hashlib.sha256(('\n'.join(KHO) + '\n').encode()).hexdigest() == D['gallery_sha256'], f'{len(KHO)} video')
check('giao kho với train của model nào cũng rỗng', COUNTS['in train of a model'] == 0, str(COUNTS))
from huggingface_hub import snapshot_download
TEACHER_DIR = snapshot_download('nguyenminh04/htstep-val-seen-feats', repo_type='dataset', allow_patterns=['omni_av/*.npz'], local_dir=f'{A}/val_feats') + '/omni_av'
TEACHER = hd.teacher_vectors(TEACHER_DIR)
DUR = {v: float(D['durations'][v]) for v in KHO}
SMOKE_VIDS = hd.smoke_videos(TEACHER, DUR, {r['video_id'] for r in VAL_ROWS if r['src_transcoded']})
VIDS = SMOKE_VIDS if SMOKE else KHO
print('vector teacher:', len(TEACHER), 'đoạn GT,', len({k[0] for k in TEACHER}), 'video', '| SMOKE:' if SMOKE else '', SMOKE_VIDS if SMOKE else '')
TRAIN_STAT = hd.train_steps_per_video(ANN)
check('số bước / video train = protocol', abs(TRAIN_STAT['mean'] - RESOLVED['train_steps_per_video']) < 1e-9, str(TRAIN_STAT))
CAPS = hd.queries(VAL_ROWS, KHO)
check('số câu hỏi = protocol', len(CAPS) == D['queries'], str(len(CAPS)))
FULL = [i for i, c in enumerate(CAPS) if c['partial'] == 'Full Match']
check('số câu Full Match = protocol', len(FULL) == D['queries_full_match'], str(len(FULL)))
GT = hd.gt_partition(CAPS)
HSTAT = hd.headline_stats(CAPS)
INV.update({'lọc video': COUNTS, 'train_ids': {k: D['train_ids'][k] for k in ('uniav_train', 'omni_train', 'both', 'omni_only', 'uniav_only')},
            'câu hỏi': len(CAPS), 'câu Full Match': len(FULL), 'text trùng': HSTAT, 'train_steps': TRAIN_STAT,
            'thời lượng (s)': {'trung vị': float(np.median(list(DUR.values()))), 'lớn nhất': max(DUR.values())}})
print(json.dumps({k: v for k, v in INV.items() if k != 'text trùng'}, indent=1, ensure_ascii=False))
print({k: v for k, v in HSTAT.items() if k != 'top'})
display(pd.DataFrame(HSTAT['top']))
wj(f'{RES}/inventory.json', INV)
done('inventory', t0, counts=COUNTS)


In [ ]:
# ===== 5. mp4 val (Drive giữ bản gốc, resume; AV1 chuyển H.264 không mất mát trên đĩa local như precut), thời lượng so với protocol =====
import importlib.util as _ilu
_spec = _ilu.spec_from_file_location('htstep_io', f'{OMNI_FIX}/src/omniretriever/data/htstep_io.py')   # chỉ file này, không kéo cả gói omniretriever
hio = _ilu.module_from_spec(_spec); _spec.loader.exec_module(hio)
t0 = time.time()
items = hd.read_jsonl(items_p)
SRC = {r['video_id']: r['source'] for r in items}
TAR = {r['video_id']: (int(r['offset_data']), int(r['size'])) for r in csv.DictReader(open(f'{OMNI_FIX}/data/htstep/tar_index.csv', newline='', encoding='utf-8'))}
check('sha256 tar_index.csv', sha256(f'{OMNI_FIX}/data/htstep/tar_index.csv') == D['tar_index']['sha256'])
def fetch_one(v):
    dst = f'{VID}/{v}.mp4'
    if os.path.exists(dst) and os.path.getsize(dst):
        return v, 'có sẵn'
    tmp = f'/content/tmp_vid/{v}'
    os.makedirs(tmp, exist_ok=True)
    try:
        s, raw = SRC[v], f'{tmp}/raw.mp4'
        if 'tar' in s:
            hio.range_get(s['tar'], *TAR[v], raw, repo=s['repo'])
        else:
            hio.hf_get(s['repo'], s['file'], raw)
        shutil.copy2(raw, dst + '.part')
        os.replace(dst + '.part', dst)
        return v, 'tải'
    except Exception as e:     # noqa: BLE001 - một video lỗi không làm mất các video khác
        return v, f'lỗi: {type(e).__name__}: {e}'
    finally:
        shutil.rmtree(tmp, ignore_errors=True)
free = shutil.disk_usage('/content').free / 2**30
check('đĩa /content đủ cho video + bản H.264 của AV1 (>= 60 GB trống)', free >= 60, f'{free:.0f} GB')
with ThreadPoolExecutor(4) as ex:
    STATUS = dict(ex.map(fetch_one, VIDS))
print(Counter(s.split(':')[0] for s in STATUS.values()))
bad = {v: s for v, s in STATUS.items() if s.startswith('lỗi')}
check('mọi video mp4 có đủ', not bad, str(list(bad.items())[:3]))

def to_local(v):
    # decord không đọc được AV1: bản đọc được nằm trên đĩa local (qp 0, nặng), Drive chỉ giữ file gốc
    src, dst = f'{VID}/{v}.mp4', f'/content/videos/{v}.mp4'
    if os.path.exists(dst) and os.path.getsize(dst):
        return v, 'có sẵn'
    p = hio.probe(src)
    if p['codec'] != 'av1':
        shutil.copy2(src, dst + '.part'); os.replace(dst + '.part', dst)
        return v, 'copy'
    ok, why, _ = hio.transcode(src, dst + '.h264.mp4', p)
    if not ok:
        return v, f'lỗi transcode: {why}'
    os.replace(dst + '.h264.mp4', dst)
    return v, 'av1 -> h264'
os.makedirs('/content/videos', exist_ok=True)
with ThreadPoolExecutor(8) as pool:
    TO_LOCAL = dict(pool.map(to_local, VIDS))
print(Counter(s.split(':')[0] for s in TO_LOCAL.values()))
bad = {v: s for v, s in TO_LOCAL.items() if s.startswith('lỗi')}
check('mp4 local đọc được bằng decord (AV1 đã chuyển H.264)', not bad, str(list(bad.items())[:3]))
def measure(v):
    p = hio.probe(f'/content/videos/{v}.mp4')
    num, den = (int(x) for x in (p['fps'] or '0/1').split('/'))
    return v, p['frames'] * den / num if p['frames'] and num else p['duration']
with ThreadPoolExecutor(8) as ex:
    MEAS = dict(ex.map(measure, VIDS))
gap = {v: abs(MEAS[v] - DUR[v]) for v in VIDS}
worst = sorted(gap, key=gap.get)[-3:]
check('thời lượng mp4 = protocol (lệch <= 0.1 s)', max(gap.values()) <= 0.1, str([(v, round(MEAS[v], 3), DUR[v]) for v in worst]))
done('videos', t0, status=dict(Counter(s.split(':')[0] for s in STATUS.values())))


In [ ]:
# ===== 6. Mục 3.1: số tái lập UniAV (chạy local: mAP, OmniCos, ArtAcc) =====
t0 = time.time()
UNIAV_DIR = f"{ROOT}/{D['uniav_cache_dir']}"
REP = PR['reproduction']['uniav_gt_article_510']
for f in ('uniav_raw.jsonl', 'checks.json'):
    check(f'có cache UniAV {f} trên Drive', os.path.exists(f'{UNIAV_DIR}/{f}'),
          f'{UNIAV_DIR}/{f}: chép thư mục D:/Học/KL/output/htstep_uemr/uniav lên Drive rồi chạy lại ô này')
CH = rj(f'{UNIAV_DIR}/checks.json')
TOL = REP['tol']
check('mAP@[0.3:0.7] (article GT, 510 video) như model card', abs(CH['map_gt']['mAP'] - REP['mAP']) <= TOL, f"{CH['map_gt']['mAP']:.2f} vs {REP['mAP']}")
check('OmniCos như model card', abs(CH['OmniCos'] - REP['OmniCos']) <= TOL, f"{CH['OmniCos']:.2f} vs {REP['OmniCos']}")
check('ArtAcc như model card', abs(CH['ArtAcc'] - REP['ArtAcc']) <= TOL, f"{CH['ArtAcc']:.2f} vs {REP['ArtAcc']}")
print('mAP article dự đoán:', CH['map_pred']['mAP'], '| so với predictions_val_seen.csv:', CH['vs_hf_csv'])
INV['uniav_reproduction'] = {k: CH[k] for k in ('videos', 'ArtAcc', 'OmniCos', 'omni_segments', 'vs_hf_csv')}
INV['uniav_map'] = {k: {x: y for x, y in CH[k].items() if x != 'stdout'} for k in ('map_gt', 'map_pred')}
wj(f'{RES}/inventory.json', INV)
done('uniav_repro', t0)


In [ ]:
# ===== 7. Cache UniAV (đoạn thô hai chế độ article): kiểm hash, không chạy lại =====
t0 = time.time()
RAW_PATH = f'{UNIAV_DIR}/uniav_raw.jsonl'
RAW_SHA = sha256(RAW_PATH)
check('sha256 uniav_raw.jsonl = protocol', RAW_SHA == D['uniav_raw_sha256'], RAW_SHA[:16])
RAW = {json.loads(l)['video_id']: json.loads(l) for l in open(RAW_PATH, encoding='utf-8') if l.strip()}
check('cache đủ kho', set(RAW) == set(KHO), f'{len(RAW)} video')
PROPS_PRED, PROPS_GT = {v: RAW[v]['pred'] for v in KHO}, {v: RAW[v]['gt'] for v in KHO}
if STUDENT:
    for r, s in D['student_sha256'].items():
        f = f'{UNIAV_DIR}/student_{r}.npz'
        check(f'sha256 student_{r}.npz = protocol', os.path.exists(f) and sha256(f) == s, 'thiếu file: chép student_*.npz từ D:/Học/KL/output/htstep_uemr/uniav lên Drive' if not os.path.exists(f) else '')
done('uniav_cache', t0, sha=RAW_SHA[:16])


In [ ]:
# ===== 8. Luật θ, L (mục 4.2), các dòng, chất lượng cắt (mục 5), đếm clip, ước lượng giờ =====
from omni_retrieval.partitions_uemr import best_tiou, coverage, segment_stats
from omni_retrieval.manifest import segment_table, unique_segments, segment_records, caption_records, store_keys, read_store, write_jsonl, seg_key
t0 = time.time()
P = {**PR['params']['r6_postprocess'], **{k: PR['params']['r8'][k] for k in ('l_min', 'l_fill')}, 'theta_grid': PR['rules']['theta']['grid']}
RS = hp.resolve(PROPS_PRED, PROPS_GT, DUR, GT, TRAIN_STAT['mean'], P)
for k in ('theta', 'mean_k', 'chunk_sec', 'clips_r8', 'clips_r2'):
    check(f'luật {k} = protocol', RS[k] == RESOLVED[k], f'{RS[k]} vs {RESOLVED[k]}')
PARTS = RS['parts']
check('các dòng = bộ đoạn đã khoá (và vector student đã tính trên đó)', hashlib.sha256(json.dumps(PARTS, sort_keys=True).encode()).hexdigest() == RESOLVED['parts_sha256'])
if SMOKE:       # luật θ, L chạy trên cả kho ở trên; từ đây chỉ 5 video
    keep = set(SMOKE_VIDS)
    KHO, CAPS = SMOKE_VIDS, [c for c in CAPS if c['video_id'] in keep]
    PARTS = {n: {v: PARTS[n][v] for v in KHO} for n in PARTS}
    GT = {v: GT[v] for v in KHO}
    FULL = [i for i, c in enumerate(CAPS) if c['partial'] == 'Full Match']
    print(f'[smoke] {len(KHO)} video, {len(CAPS)} câu')
for n in ROWS:
    check(f'{LABEL[n]}: mọi video có đoạn', all(len(PARTS[n][v]) for v in KHO))
FILE_SHA = {}
for n in ROWS:
    p = f'{SEGDIR}/{hp.FILES[n]}.json'
    wj(p, {v: PARTS[n][v] for v in KHO})
    FILE_SHA[hp.FILES[n]] = sha256(p)
SEG_HASH = hashlib.sha256(json.dumps(FILE_SHA, sort_keys=True).encode()).hexdigest()
wj(f'{SEGDIR}/SOURCE.json', {'protocol_sha16': PH[:16], 'seg_hash': SEG_HASH, 'files': FILE_SHA, 'commit': COMMIT, 'created': time.strftime('%Y-%m-%d %H:%M:%S')})
print('SEG_HASH:', SEG_HASH, '| θ', RS['theta'], '| L', RS['chunk_sec'], '| K mean', round(RS['mean_k'], 3))

def quality(name):
    r, g = {v: PARTS[name][v] for v in KHO}, {v: GT[v] for v in KHO}
    bt = best_tiou(r, g)
    kp, kg = np.array([len(r[v]) for v in KHO]), np.array([len(g[v]) for v in KHO])
    rank = lambda a: pd.Series(a).rank().to_numpy()
    lens = np.array([te - ts for v in KHO for ts, te in r[v]]); gl = np.array([te - ts for v in KHO for ts, te in g[v]])
    return {'bước GT': int(len(bt)), 'recall': {f'tIoU {t}': 100 * float((bt >= t).mean()) for t in (0.3, 0.5, 0.7)},
            'bỏ sót % (mọi đoạn tIoU < 0.3)': 100 * float((bt < 0.3).mean()), 'tIoU tốt nhất trung bình': float(bt.mean()),
            'K_pred trung vị': float(np.median(kp)), 'K_GT trung vị': float(np.median(kg)), 'K_pred / K_GT trung vị': float(np.median(kp / kg)),
            'tương quan K': {'pearson': float(np.corrcoef(kp, kg)[0, 1]), 'spearman': float(np.corrcoef(rank(kp), rank(kg))[0, 1])},
            'độ phủ %': 100 * float(np.mean([coverage(r[v], DUR[v]) for v in KHO])), 'độ phủ GT %': 100 * float(np.mean([coverage(g[v], DUR[v]) for v in KHO])),
            'độ dài đoạn trung vị (s)': float(np.median(lens)), 'độ dài bước GT trung vị (s)': float(np.median(gl))}
Q = {'R6': quality('uniav_pred'), 'R6gt': quality('uniav_gtart'), 'ArtAcc %': CH['ArtAcc'], 'mAP article GT': CH['map_gt']['mAP'], 'mAP article dự đoán': CH['map_pred']['mAP'],
     'R6gt - R6: giá của việc không biết article (recall tIoU 0.5, điểm)': quality('uniav_gtart')['recall']['tIoU 0.5'] - quality('uniav_pred')['recall']['tIoU 0.5']}
wj(f'{RES}/uniav_quality.json', Q)
STATS = pd.DataFrame([{'dòng': LABEL[n], **segment_stats(PARTS[n], {v: DUR[v] for v in KHO})} for n in ROWS]).round(2)
if not SMOKE:
    print(json.dumps(Q, indent=1, ensure_ascii=False))
    display(STATS)
SEG_ROWS = segment_table(PARTS)
UNIQ = unique_segments(SEG_ROWS)
cnt = pd.DataFrame([{'dòng': LABEL[n], 'đoạn': sum(len(PARTS[n][v]) for v in KHO), 'vec/video': np.mean([len(PARTS[n][v]) for v in KHO])} for n in ROWS])
display(cnt.round(2))
SEC_REF = 0.316     # s/clip Omni LoRA trên A100 (nhật ký: eval val 315.6 ms/item); ô 10 đo lại 20 clip
print(f'clip khác nhau: {len(UNIQ)} | câu hỏi: {len(CAPS)} | ước lượng A100: {len(UNIQ) * SEC_REF / 3600:.2f} giờ / tag')
INV['clips'] = {'unique': len(UNIQ), 'by_row': cnt.to_dict('records'), 'seg_hash': SEG_HASH}
wj(f'{RES}/inventory.json', INV)
done('partitions', t0, unique_clips=len(UNIQ), seg_hash=SEG_HASH)


In [ ]:
# ===== 9. Mục 3.2: thầy có trùng best không; encode 20 clip GT phải trùng val_embeds_best.npz (cos >= 0.999) =====
from omni_retrieval.collect import collect_segments, collect_text, local_npz
from omni_retrieval.manifest import PROMPT
t0 = time.time()
BASE_MODEL, BEATS = '/content/WAVE_HOME/WAVE-7B', '/content/WAVE_HOME/BEATs_iter3_plus_AS2M_finetuned_on_AS2M_cpt2.pt'
OM = lambda tag: f'{H}/omni/{tag}'
ADAPTERS = {'best': BEST_ADAPTER, 'pt': PT_ADAPTER}

def wave_ok():
    idx = glob.glob(f'{BASE_MODEL}/*.index.json')
    return os.path.exists(BEATS) and bool(idx) and all(os.path.exists(f'{BASE_MODEL}/{s}') for s in set(json.load(open(idx[0]))['weight_map'].values()))

def omni_env(tag):
    check('GPU đủ cho Omni 7B (>= 22 GB)', GPU_GB >= 22, f'{gpu}, {GPU_GB:.0f} GB')
    chk = subprocess.run([sys.executable, '-c', 'import decord, av, peft, torch, transformers; print(transformers.__version__, torch.cuda.is_available())'], capture_output=True, text=True)
    last = (chk.stdout.strip().splitlines() or [''])[-1]
    check('môi trường encode (transformers 4.51.3, CUDA)', chk.returncode == 0 and last.split()[:2] == ['4.51.3', 'True'], f'{last} {chk.stderr[-300:]}')
    if not wave_ok():
        sh('hf download nguyenminh04/omni-model --local-dir /content/WAVE_HOME --max-workers 2')
    check('WAVE-7B + BEATs đủ file', wave_ok())
    adapter = ADAPTERS[tag]
    from safetensors import safe_open
    with safe_open(f'{adapter}/adapter_model.safetensors', 'pt') as f:
        keys = list(f.keys())
    heads = [h for h in ('classify_linear', 'beats_ln', 'beats_proj') if any(h in k and 'lora_' not in k for k in keys)]
    check(f'{tag}: adapter có fusion head', len(heads) == 3, str(heads))
    return f'cd {FUSHION} && python -m omni_retrieval.encode --base-model {BASE_MODEL} --beats-path {BEATS} --adapter {adapter} --video-root /content/videos'

def encode(modality, manifest, store, log, enc, extra=''):
    # chạy encode; trả (id còn thiếu, tổng id). | tee che mã lỗi của tiến trình con nên tự đếm lại từ store
    with open(log, 'a') as f:
        f.write(f"\n===== {modality} | {time.strftime('%Y-%m-%d %H:%M:%S')} =====\n")
    get_ipython().system(f'{enc} --modality {modality} --manifest "{manifest}" --store "{store}" {extra} 2>&1 | tee -a "{log}"')
    todo = [json.loads(l)['id'] for l in open(manifest) if l.strip()]
    return sorted(set(todo) - store_keys(store, modality)), len(todo)

T = PR['models']['omni_teacher']
best_lora = f'{RUN_DRIVE}/ckpt/best/lora_only.safetensors'
same = os.path.exists(best_lora) and sha256(best_lora) == T['sha256']
print('thầy của UniAV (omni-adapters/checkpoint) trùng best của user:', same)
check('best khác thầy (lý do cột (S) dùng text của thầy)', not same, 'best trùng thầy: sửa protocol rồi chạy lại')
INV['teacher_vs_best'] = {'teacher_lora_sha256': T['sha256'], 'teacher_adapter_sha256': T['adapter_sha256'], 'best_adapter_sha256': PR['models']['omni_best']['sha256'],
                          'best_lora_sha256': sha256(best_lora) if os.path.exists(best_lora) else 'không có trên Drive', 'same': same}
wj(f'{RES}/inventory.json', INV)

EMB = f'{RUN_DRIVE}/eval/val_embeds_best.npz'
check('có val_embeds_best.npz trên Drive', os.path.exists(EMB), EMB)
rng = np.random.default_rng(SEED)
with np.load(EMB) as z:
    ids = z['ids'].astype(str)
    _v = set(VIDS)
    pool = [i for i, x in enumerate(ids) if x.rsplit('#', 1)[0] in _v]
    check('đủ 20 clip GT trong kho đang chạy', len(pool) >= 20, str(len(pool)))
    pick = sorted(rng.choice(pool, 20, replace=False).tolist())
    REF = {ids[i]: {'text': z['text'][i].astype(np.float32), 'av': z['av'][i].astype(np.float32)} for i in pick}
BY_ID = {r['id']: r for r in VAL_ROWS}
probe = f'{LOCAL}/probe_enc'
shutil.rmtree(probe, ignore_errors=True); os.makedirs(probe)
write_jsonl(f'{probe}/segs.jsonl', [{'id': r, 'type': 'retrieval', 'conversations': [{'from': 'human', 'value': PROMPT}], 'video': BY_ID[r]['video'], 'timestamps': BY_ID[r]['window']} for r in REF])
write_jsonl(f'{probe}/caps.jsonl', [{'id': r, 'text': BY_ID[r]['text']} for r in REF])
missing = [v for v in {BY_ID[r]['video_id'] for r in REF} if v not in KHO]
check('20 clip đều thuộc kho', not missing, str(missing))
if RUN_OMNI:
    enc = omni_env('best')
    left, _ = encode('av', f'{probe}/segs.jsonl', f'{probe}/av', f'{probe}/encode.log', enc, '--num-workers 0')
    left_t, _ = encode('text', f'{probe}/caps.jsonl', f'{probe}/text', f'{probe}/encode.log', enc, '--batch-size 8')
    check('encode đủ 20 clip + 20 text', not left and not left_t, str(left + left_t))
    fa, ft = read_store(f'{probe}/av', 'av'), read_store(f'{probe}/text', 'text')
    cosv = lambda a, b: float(a @ b / np.linalg.norm(a) / np.linalg.norm(b))
    cav = {r: cosv(fa[r], REF[r]['av']) for r in REF}; ctx = {r: cosv(ft[r], REF[r]['text']) for r in REF}
    MIN = PR['reproduction']['omni_clip_cos_min']
    print('cos AV: min %.5f mean %.5f | cos text: min %.5f mean %.5f' % (min(cav.values()), np.mean(list(cav.values())), min(ctx.values()), np.mean(list(ctx.values()))))
    INV['omni_encode_check'] = {'cos_av_min': min(cav.values()), 'cos_text_min': min(ctx.values()), 'n': len(REF)}
    wj(f'{RES}/inventory.json', INV)
    check('encode clip = val_embeds_best (AV, cos >= 0.999)', min(cav.values()) >= MIN, str({k: round(v, 4) for k, v in cav.items() if v < MIN}))
    check('encode text = val_embeds_best (cos >= 0.999)', min(ctx.values()) >= MIN, str({k: round(v, 4) for k, v in ctx.items() if v < MIN}))

caps_rec = [{'caption_id': c['caption_id'], 'sentence': c['sentence']} for c in CAPS]
if RUN_OMNI and STUDENT:      # MỘT lần nạp Omni thầy (venv riêng): encode_av 20 clip GT phải trùng omni_av/, rồi mới encode_text mọi câu hỏi
    VENV = '/content/venv_teacher'
    if not os.path.exists(f'{VENV}/bin/python'):
        sh(f'{sys.executable} -m venv --system-site-packages --without-pip {VENV}')
        sh(f'{sys.executable} -m pip --python {VENV}/bin/python install -q transformers==4.57.3 peft==0.20.0 av')
    v = subprocess.run([f'{VENV}/bin/python', '-c', 'import transformers, peft, av, torch; print(transformers.__version__, peft.__version__, torch.cuda.is_available())'], capture_output=True, text=True)
    check('môi trường thầy (transformers 4.57.3, peft 0.20.0, CUDA)', v.stdout.split()[:3] == ['4.57.3', '0.20.0', 'True'], v.stdout + v.stderr[-300:])
    if not wave_ok():
        sh('hf download nguyenminh04/omni-model --local-dir /content/WAVE_HOME --max-workers 2')
    check('WAVE-7B + BEATs đủ file', wave_ok())
    _v = set(VIDS)
    pool = sorted(k for k in TEACHER if k[0] in _v)
    check('đủ 20 clip có vector teacher', len(pool) >= 20, str(len(pool)))
    chosen = [pool[i] for i in sorted(rng.choice(len(pool), 20, replace=False).tolist())]
    TD = OM('student_text')
    os.makedirs(TD, exist_ok=True)
    clips = [{'id': '|'.join(map(str, k)), 'video': f'{VID}/{k[0]}.mp4', 'start': TEACHER[k]['segment'][0], 'end': TEACHER[k]['segment'][1],
              'npz': TEACHER[k]['npz'], 'row': TEACHER[k]['row']} for k in chosen]
    wj(f'{TD}/clips.json', clips)
    write_jsonl(f'{TD}/caps.jsonl', caption_records(caps_rec, set()))
    if os.path.exists(f'{TD}/report.json'):
        os.remove(f'{TD}/report.json')
    r = subprocess.run([f'{VENV}/bin/python', '-m', 'uemr_htstep.teacher_encode', '--base-model', BASE_MODEL, '--adapter', TEACHER_ADAPTER, '--clips', f'{TD}/clips.json',
                        '--captions', f'{TD}/caps.jsonl', '--store', f'{TD}/text', '--report', f'{TD}/report.json', '--min-cos', str(PR['reproduction']['teacher_clip_cos_min'])],
                       cwd=FUSHION, capture_output=True, text=True, env={**os.environ, 'PYTHONPATH': f'{OMNI_AV}/src:{FUSHION}', 'WAVE_HOME': '/content/WAVE_HOME', 'BEATS_PATH': BEATS})
    print(r.stdout[-1500:]); print(r.stderr[-800:] if r.returncode else '')
    REP_T = rj(f'{TD}/report.json', {})
    check('adapter thầy: encode_av 20 clip GT = omni_av/ (cos >= 0.999)', r.returncode == 0 and REP_T.get('ok') is True, str({k: REP_T.get(k) for k in ('n', 'min', 'mean', 'below')}))
    os.makedirs(f'{TD}/db', exist_ok=True)
    collect_text([{k: c[k] for k in ('caption_id', 'video_id', 'sentence', 'gt_ts', 'gt_te', 'gt_seg_key')} for c in CAPS], f'{TD}/text', f'{TD}/db/text.npz')
    INV['teacher_encode_check'] = {k: REP_T.get(k) for k in ('n', 'min', 'mean', 'sec_per_clip')}
    wj(f'{RES}/inventory.json', INV)
done('omni_check', t0)


In [ ]:
# ===== 10. Omni encode mọi clip và câu hỏi cho từng tag (resume theo store) =====
def failures(store, ids):
    err = {}
    if os.path.exists(f'{store}/failed.jsonl'):
        for l in open(f'{store}/failed.jsonl'):
            if l.strip():
                q = json.loads(l); err[q['id']] = q['error']
    return [{'seg_key': k, 'error': err.get(k, 'không có trong failed.jsonl')} for k in ids]
OMNI_INFO = rj(f'{H}/omni/info.json', {})
if RUN_OMNI:
    for tag in OMNI_TAGS:
        t0 = time.time()
        AV, TX, DBD = f'{OM(tag)}/clips', f'{OM(tag)}/text', f'{OM(tag)}/db'
        log = f'{OM(tag)}/encode.log'
        os.makedirs(OM(tag), exist_ok=True)
        meta = rj(f'{DBD}/meta.json', {})
        av_todo = segment_records(UNIQ, store_keys(AV, 'av'))
        tx_todo = caption_records(caps_rec, store_keys(TX, 'text'))
        if meta.get('seg_hash') == SEG_HASH and meta.get('protocol_sha16') == PH[:16] and not tx_todo and len(av_todo) == len(meta.get('failed', [])):
            print(f'[{tag}] đã encode và gom đủ: bỏ qua'); continue
        enc = omni_env(tag)
        print(f'[{tag}] clip cần encode: {len(av_todo)} / {len(UNIQ)} | câu hỏi cần encode: {len(tx_todo)}')
        info = {}
        if av_todo:
            write_jsonl(f'{OM(tag)}/segs_todo.jsonl', av_todo)
            if len(av_todo) > 20:
                encode('av', f'{OM(tag)}/segs_todo.jsonl', AV, log, enc, '--num-workers 4 --limit 20')
                spc = json.loads(open(f'{AV}/runs.jsonl').read().strip().splitlines()[-1]).get('sec_per_clip') or float('nan')
                info['sec_per_clip_20'] = spc
                print(f'[{tag}] 20 clip: {spc:.3f} s/clip -> còn {len(av_todo) - 20} clip ≈ {(len(av_todo) - 20) * spc / 3600:.2f} giờ')
            encode('av', f'{OM(tag)}/segs_todo.jsonl', AV, log, enc, '--num-workers 4')
        left = [r['id'] for r in segment_records(UNIQ, store_keys(AV, 'av'))]
        FAIL = failures(AV, left)
        pd.DataFrame(FAIL, columns=['seg_key', 'error']).to_csv(f'{OM(tag)}/failed.csv', index=False)
        fail_pct = 100 * len(FAIL) / len(UNIQ)
        check(f'{tag}: clip lỗi <= 2%', fail_pct <= 2, f'{len(FAIL)} clip ({fail_pct:.2f}%), xem {OM(tag)}/failed.csv')
        if tx_todo:
            write_jsonl(f'{OM(tag)}/caps_todo.jsonl', tx_todo)
            left_t, _ = encode('text', f'{OM(tag)}/caps_todo.jsonl', TX, log, enc, '--batch-size 32')
            check(f'{tag}: encode đủ câu hỏi', not left_t, str(left_t[:5]))
        L = f'{LOCAL}/htdb_{tag}'
        os.makedirs(L, exist_ok=True)
        seg = collect_segments(SEG_ROWS, AV, f'{L}/segments.npz', allow_missing=True)
        txt = collect_text([{k: c[k] for k in ('caption_id', 'video_id', 'sentence', 'gt_ts', 'gt_te', 'gt_seg_key')} for c in CAPS], TX, f'{L}/text.npz')
        os.makedirs(DBD, exist_ok=True)
        for f in ('segments.npz', 'text.npz'):
            shutil.copy2(f'{L}/{f}', f'{DBD}/{f}')
        meta = {'tag': tag, 'seg_hash': SEG_HASH, 'protocol_sha16': PH[:16], 'segments': seg, 'text': txt, 'failed': [f['seg_key'] for f in FAIL], **info,
                'adapter_sha256': sha256(f'{ADAPTERS[tag]}/adapter_model.safetensors'), 'commit': COMMIT, 'created': time.strftime('%Y-%m-%d %H:%M:%S'),
                'sha256': {f: sha256(f'{L}/{f}') for f in ('segments.npz', 'text.npz')}}
        wj(f'{DBD}/meta.json', meta)
        OMNI_INFO[tag] = meta
        wj(f'{H}/omni/info.json', OMNI_INFO)
        print(f'[{tag}] segments {seg} | text {txt}')
        done(f'encode_{tag}', t0, failed=len(FAIL))
        torch.cuda.empty_cache()


In [ ]:
# ===== 11. Chấm từng tag: mọi dòng x cột, strict + multi =====
from omni_retrieval.evaluate import load_text, paired_bootstrap, rank_metrics, text_to_clip as yc2_text_to_clip, video_ranks
from omni_retrieval.search import SegmentIndex
from uemr_htstep import metrics as hm
MODES = hm.MODES

def boot(a, b, vids, k=1):
    bs = paired_bootstrap(a, b, k, NB, BSEED, 'video', vids)
    return {'A - B': bs['diff'], 'CI': f"[{bs['ci_low']:+.2f}, {bs['ci_high']:+.2f}]", 'ci_low': bs['ci_low'], 'ci_high': bs['ci_high']}
def tag_ready(tag):
    m = rj(f'{OM(tag)}/db/meta.json', {})
    return m.get('seg_hash') == SEG_HASH and m.get('protocol_sha16') == PH[:16]
def ranks_path(tag, row, col='b'):
    return f'{RES}/ranks_{tag}_{col}_{row}.json'
PAIRS = [('r8', 'global'), ('r8', 'uni_M'), ('uniav_pred', 'uni_M'), ('r8', 'uni_chunk'), ('uniav_pred', 'global'), ('r8', 'uniav_pred'), ('uniav_gtart', 'uniav_pred'), ('gt', 'r8')]
VCMR_PAIRS = [('r8', 'uni_M'), ('uniav_pred', 'uni_M'), ('r8', 'uniav_pred')]
FULL_IDX = np.array(FULL)
if RUN_EVAL:
    for tag in OMNI_TAGS:
        if not tag_ready(tag):
            print(f'[{tag}] chưa có vector với SEG_HASH này: bỏ qua'); continue
        t0 = time.time()
        DB = str(local_npz(f'{OM(tag)}/db', f'{LOCAL}/htdb_{tag}', ('segments.npz', 'text.npz'), ROWS))
        text = load_text(DB)
        check(f'{tag}: text.npz cùng thứ tự câu hỏi', [str(x) for x in text['caption_id']] == [c['caption_id'] for c in CAPS])
        vids = text['video_id']
        R, TAB = {}, []
        for p in ROWS:
            idx = SegmentIndex.load(DB, p)
            res = {m: hm.text_to_video(idx, text, CAPS, m) for m in MODES}
            check(f'{tag}/{LABEL[p]}: strict = omni_retrieval.evaluate.video_ranks', np.array_equal(res['strict']['ranks'], video_ranks(idx, text)['ranks']))
            R[p] = res
            if SMOKE:
                print(f"[smoke] {tag}/{LABEL[p]}: emb {idx.emb.shape}, text {text['emb'].shape}")
            for m in MODES:
                TAB.append({'tag': tag, 'mode': m, 'cột': 'b', 'dòng': LABEL[p], **rank_metrics(res[m]['ranks']), 'VCMR R@1': float('nan') if p == 'global' else 100 * float(res[m]['vcmr'].mean()),
                            'R@1 Full Match': 100 * float((res[m]['ranks'][FULL_IDX] <= 1).mean()), 'vec/video': float(np.mean(idx.counts)), 'video có vector': int(idx.n_videos)})
            wj(ranks_path(tag, LABEL[p]), {'caption_id': [c['caption_id'] for c in CAPS], 'video_id': [str(x) for x in vids], **{m: res[m]['ranks'].tolist() for m in MODES},
                                           **{f'vcmr_{m}': res[m]['vcmr'].astype(int).tolist() for m in MODES}})
        ck, ce = hm.gt_clips(f'{DB}/segments.npz')
        T2C = {m: hm.text_to_clip(text, CAPS, ck, ce, m) for m in MODES}
        yc2 = yc2_text_to_clip(DB)
        check(f'{tag}: text -> clip strict = omni_retrieval.evaluate.text_to_clip (kho clip GT)', all(abs(rank_metrics(T2C['strict'])[k] - yc2['subset'][k]) < 1e-9 for k in ('R@1', 'R@5', 'R@10', 'MedR')))
        if not SMOKE:
            print(f"[{tag}] text -> clip (đường chéo {len(CAPS)} câu = eval_htstep): R@1 {yc2['eval']['R@1']:.3f}")
        if tag == 'best' and not SMOKE:
            ref, tol = EV['omni_t2v_R@1_best'], EV['omni_t2v_R@1_tol']
            check('text -> clip đường chéo = metrics_htstep_best.json', abs(yc2['eval']['R@1'] - ref) <= tol, f"{yc2['eval']['R@1']:.3f} vs {ref}, cho phép {tol}")
        for m in MODES:
            TAB.append({'tag': tag, 'mode': m, 'cột': 'b', 'dòng': 'text -> clip (kho clip GT)', **rank_metrics(T2C[m]), 'VCMR R@1': float('nan'), 'R@1 Full Match': float('nan'), 'vec/video': float('nan'), 'video có vector': len(ck)})
        wj(ranks_path(tag, 't2c'), {'caption_id': [c['caption_id'] for c in CAPS], 'video_id': [str(x) for x in vids], **{m: T2C[m].tolist() for m in MODES}})

        if STUDENT:
            ST = str(local_npz(f"{OM('student_text')}/db", f'{LOCAL}/htdb_student', ('text.npz',)))
            ts_ = load_text(ST)
            check('text student cùng thứ tự', [str(x) for x in ts_['caption_id']] == [c['caption_id'] for c in CAPS])
            stu_file = lambda n: f"{UNIAV_DIR}/student_{hp.FILES[n]}.npz"
            r6_key = {hm.row_key(v, a, b) for i6 in [SegmentIndex.load(DB, 'uniav_pred')] for v, a, b in zip(i6.seg_video, i6.ts, i6.te)}
            means, means_b = {}, {}
            for col in ('S', 'U', 'C1'):
                for p in ROWS:
                    if col == 'C1' and p != 'r8':
                        continue
                    idx = SegmentIndex.load(DB, p)
                    with np.load(stu_file(p)) as z:
                        e_s = hm.student_rows(idx, z['video_id'], z['ts'], z['te'], z['emb'])
                    fill = np.array([hm.row_key(v, a, b) not in r6_key for v, a, b in zip(idx.seg_video, idx.ts, idx.te)])
                    sim = hm.column_sim(col, text['emb'], ts_['emb'], idx.emb, e_s, BETA, fill)
                    if col == 'S':
                        sb = text['emb'] @ idx.emb.T
                        means[LABEL[p]], means_b[LABEL[p]] = {'mean': float(sim.mean()), 'std': float(sim.std())}, {'mean': float(sb.mean()), 'std': float(sb.std())}
                    res = {m: hm.text_to_video(idx, text, CAPS, m, sim=sim) for m in MODES}
                    R[f'{col}:{p}'] = res
                    for m in MODES:
                        TAB.append({'tag': tag, 'mode': m, 'cột': col, 'dòng': LABEL[p], **rank_metrics(res[m]['ranks']), 'VCMR R@1': float('nan') if p == 'global' else 100 * float(res[m]['vcmr'].mean()),
                                    'R@1 Full Match': 100 * float((res[m]['ranks'][FULL_IDX] <= 1).mean()), 'vec/video': float(np.mean(idx.counts)), 'video có vector': int(idx.n_videos)})
            if SMOKE:
                print(f"[smoke] student: text thầy {ts_['emb'].shape}, cột S / U / C1 chấm được")
            else:
                print('phân bố điểm cột (b):', json.dumps(means_b)); print('phân bố điểm cột (S):', json.dumps(means))

        REF = None
        if STUDENT and tag == 'best':       # tham chiếu: text thầy · vector teacher so với text thầy · vector student, trên cùng clip GT
            keep_q, by_clip = hd.teacher_queries(CAPS, VAL_ROWS, TEACHER)
            with np.load(f"{UNIAV_DIR}/student_{hp.FILES['gt']}.npz") as z:
                stu = {hm.row_key(v, a, b): e for v, a, b, e in zip(z['video_id'], z['ts'], z['te'], z['emb'])}
            caps_r = [CAPS[i] for i in keep_q]
            clip_of = {c['gt_seg_key']: c for c in caps_r}
            keys = sorted(by_clip)
            unit = lambda a: a / np.linalg.norm(a, axis=1, keepdims=True)
            e_t = unit(np.stack([by_clip[k] for k in keys]).astype(np.float32))
            e_s = unit(np.stack([stu[hm.row_key(clip_of[k]['video_id'], clip_of[k]['gt_ts'], clip_of[k]['gt_te'])] for k in keys]).astype(np.float32))
            sub_t = {'emb': ts_['emb'][keep_q], 'gt_seg_key': np.array([c['gt_seg_key'] for c in caps_r])}
            RA, RB = ({m: hm.text_to_clip(sub_t, caps_r, keys, e, m) for m in MODES} for e in (e_t, e_s))
            vr = [c['video_id'] for c in caps_r]
            REF = {'câu': len(caps_r), 'clip': len(keys), 'video': len(set(vr)), 'bảng': [{'mode': m, 'vector': n, **rank_metrics(r[m])} for m in MODES for n, r in (('teacher', RA), ('student', RB))],
                   'teacher - student': {m: boot(RA[m], RB[m], vr) for m in MODES}}
            if SMOKE:
                print(f'[smoke] tham chiếu: teacher {e_t.shape}, student {e_s.shape}, câu {len(caps_r)}')
            else:
                print('=== tham chiếu: text thầy -> clip GT (student kém thầy bao nhiêu) ==='); display(pd.DataFrame(REF['bảng']).round(2)); print(json.dumps(REF['teacher - student'], ensure_ascii=False))

        PB = []
        for m in MODES:
            for a, b in PAIRS:
                PB.append({'tag': tag, 'mode': m, 'chỉ số': 'R@1', 'cột': 'b', 'cặp': f'{LABEL[a]} - {LABEL[b]}', **boot(R[a][m]['ranks'], R[b][m]['ranks'], vids)})
            for a, b in VCMR_PAIRS:
                PB.append({'tag': tag, 'mode': m, 'chỉ số': 'VCMR R@1', 'cột': 'b', 'cặp': f'{LABEL[a]} - {LABEL[b]}', **boot(np.where(R[a][m]['vcmr'], 1, 2), np.where(R[b][m]['vcmr'], 1, 2), vids)})
            if STUDENT:
                PB.append({'tag': tag, 'mode': m, 'chỉ số': 'R@1', 'cột': 'U-b', 'cặp': '(U) - (b) R8', **boot(R['U:r8'][m]['ranks'], R['r8'][m]['ranks'], vids)})
                PB.append({'tag': tag, 'mode': m, 'chỉ số': 'R@1', 'cột': 'C1-R6', 'cặp': 'C1 - R6', **boot(R['C1:r8'][m]['ranks'], R['uniav_pred'][m]['ranks'], vids)})
                PB.append({'tag': tag, 'mode': m, 'chỉ số': 'R@1', 'cột': 'S-b', 'cặp': '(S) - (b) R8', **boot(R['S:r8'][m]['ranks'], R['r8'][m]['ranks'], vids)})
        tab, pb = pd.DataFrame(TAB), pd.DataFrame(PB)
        if SMOKE:
            print(f'[smoke] {tag}: bảng {tab.shape}, cặp bootstrap {pb.shape}')
        else:
            print(f'=== {tag}: multi-positive (chính) ==='); display(tab[tab['mode'] == 'multi'].drop(columns=['tag', 'mode']).round(2))
            print(f'=== {tag}: strict ==='); display(tab[tab['mode'] == 'strict'].drop(columns=['tag', 'mode']).round(2))
            display(pb.drop(columns=['ci_low', 'ci_high', 'tag']).round(2))
        wj(f'{RES}/metrics_{tag}.json', {'tag': tag, 'seg_hash': SEG_HASH, 'protocol_sha16': PH[:16], 'queries': len(CAPS), 'videos': len(KHO), 'student': STUDENT,
                                         'omni': {k: OMNI_INFO.get(tag, {}).get(k) for k in ('adapter_sha256', 'failed')}, 'table': TAB, 'pairs': PB, 'reference_teacher_student': REF, 'bootstrap': EV['bootstrap'], 'commit': COMMIT})
        done(f'eval_{tag}', t0)


In [ ]:
# ===== 12. Tiêu chí mục 7 -> criteria.json =====
t0 = time.time()
def pair_of(tag, cap, mode='multi', what='R@1', col='b'):
    m = rj(f'{RES}/metrics_{tag}.json')
    if m is None or m['seg_hash'] != SEG_HASH:
        return None
    return next((p for p in m['pairs'] if p['mode'] == mode and p['chỉ số'] == what and p['cặp'] == cap and p['cột'] == col), None)
CRIT = {'protocol_sha16': PH[:16], 'seg_hash': SEG_HASH, 'main_cell': EV['main_cell'], 'student_enabled': STUDENT}
T = {}
for name, cap in (('T1', 'R8 - R0'), ('T2', 'R8 - R3'), ('T3', 'R6 - R3')):
    p = pair_of('best', cap)
    T[name] = None if p is None else {'cặp': cap, 'A - B': p['A - B'], 'CI': p['CI'], 'ci_low': p['ci_low'], 'đạt': bool(p['ci_low'] > 0)}
if STUDENT:
    for name, cap, col in (('T4', '(U) - (b) R8', 'U-b'), ('T5', 'C1 - R6', 'C1-R6')):
        p = pair_of('best', cap, col=col)
        T[name] = None if p is None else {'cặp': cap, 'A - B': p['A - B'], 'CI': p['CI'], 'ci_low': p['ci_low'], 'đạt': bool(p['ci_low'] > 0)}
else:
    T['T4'] = T['T5'] = 'không chạy: thầy UniAV khác best, chờ quyết định của user (protocol.student)'
T6 = {}
for cap, col in (('R8 - R2', 'b'), ('R6gt - R6', 'b')) + ((('(S) - (b) R8', 'S-b'),) if STUDENT else ()):
    p = pair_of('best', cap, col=col)
    T6[cap] = None if p is None else {'A - B': p['A - B'], 'CI': p['CI']}
T['T6'] = T6
for k in ('T1', 'T2', 'T3', 'T4', 'T5', 'T6'):
    T[k] = {'test': CR[k]['test'], 'kết quả': T[k], **({'câu nếu đạt': CR[k]['sentence']} if 'sentence' in CR[k] else {})}
CRIT['criteria'] = T
if T['T1']['kết quả'] is None:
    CRIT['conclusion'] = 'chưa có kết quả Omni best (ô chính): chạy ô 9-11 với tag best'
else:
    t1, t2 = T['T1']['kết quả']['đạt'], T['T2']['kết quả']['đạt']
    C = PR['conclusion']
    CRIT['conclusion'] = C[0]['text'] if t1 and t2 else C[1]['text'] if t1 else C[2]['text']
if SMOKE:
    print('[smoke] criteria:', sorted(T))
else:
    print(json.dumps(CRIT, indent=1, ensure_ascii=False))
wj(f'{RES}/criteria.json', CRIT)
done('criteria', t0, conclusion=CRIT['conclusion'])


In [ ]:
# ===== 13. Xuất kết quả; bảng pt | best | best - pt nếu có pt =====
import zipfile
t0 = time.time()
if 'pt' in OMNI_TAGS and not SMOKE and all(rj(f'{RES}/metrics_{t}.json') for t in ('pt', 'best')):
    CMP = []
    for m in MODES:
        for name, row in (('text -> clip R@1', 't2c'), ('R@1 R0', 'R0'), ('R@1 R3', 'R3'), ('R@1 R6', 'R6'), ('R@1 R8', 'R8')):
            jb, jp = rj(ranks_path('best', row)), rj(ranks_path('pt', row))
            check(f'{name}: cùng câu hỏi', jb['caption_id'] == jp['caption_id'])
            b, p = np.array(jb[m]), np.array(jp[m])
            r = boot(b, p, jb['video_id'])
            CMP.append({'mode': m, 'chỉ số': name, 'pt': 100 * float((p <= 1).mean()), 'best': 100 * float((b <= 1).mean()), 'best - pt': r['A - B'], 'CI': r['CI']})
    cmp_ = pd.DataFrame(CMP)
    print('=== multi-positive ==='); display(cmp_[cmp_['mode'] == 'multi'].drop(columns=['mode']).round(2))
    wj(f'{RES}/compare_pt_best.json', {'seg_hash': SEG_HASH, 'protocol_sha16': PH[:16], 'rows': CMP, 'bootstrap': EV['bootstrap']})
if SMOKE:
    check('smoke: đủ file kết quả', all(os.path.exists(f'{RES}/{f}') for f in ('metrics_best.json', 'criteria.json', 'inventory.json', 'uniav_quality.json')))
    done('smoke', t0, videos=len(KHO))
    shutil.rmtree(H); shutil.rmtree(LOCAL, ignore_errors=True)
    print('SMOKE XONG: đã xoá smoke/. Đổi SMOKE = False ở ô 0 rồi Run all để chạy cả kho.')
else:
    files = sorted(glob.glob(f'{RES}/*.json'))
    ZIP = f'/content/uemr_htstep_{PROTO_SHA16[:8]}.zip'
    with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
        for f in files + [f'{SEGDIR}/SOURCE.json', MAN_PATH, f'{UNIAV_DIR}/checks.json']:
            if os.path.exists(f):
                z.write(f, os.path.relpath(f, H))
    print('kết quả trên Drive:', RES)
    for f in files:
        print(f'  {os.path.basename(f):32s} {os.path.getsize(f):>10d} B  sha256 {sha256(f)[:16]}')
    print('zip để tải về (chép vào Fushion results/uemr_htstep/%s/):' % PROTO_SHA16[:8], ZIP)
    SUM = {'protocol': PH[:16], 'commit': COMMIT, 'seg_hash': SEG_HASH, 'kho': len(KHO), 'câu hỏi': len(CAPS), 'student': STUDENT,
           'tags có kết quả': [t for t in ('pt', 'best') if os.path.exists(f'{RES}/metrics_{t}.json')],
           'kết luận': (rj(f'{RES}/criteria.json') or {}).get('conclusion'), 'thời gian phiên (phút)': round((time.time() - T_START) / 60, 1)}
    print(json.dumps(SUM, indent=1, ensure_ascii=False))
    done('export', t0, **SUM)
    try:
        from google.colab import files as _f
        _f.download(ZIP)
    except Exception as e:     # noqa: BLE001
        print('tải zip thủ công:', e)
